# DCAMF-Net 信号分析
跟一段水下噪声，看模型如何一步步把它变干净。每节回答一个问题。

## 0. 加载模型 + 样本

In [ ]:
import os, sys, json
import torch, numpy as np, soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import welch, spectrogram

os.environ['OMP_NUM_THREADS'] = '8'
PROJECT_ROOT = Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'dcamf_net'))

from dcamf_net.model import DCAMFNet
from argparse import Namespace

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

args = Namespace(enc_channels=256, enc_kernel_size=80, enc_stride=40,
                 chunk_size=500, hop_size=250, n_blocks=10, n_heads=8,
                 ffn_hidden=512, dw_kernel_size=31, checkpoint=None)

model = DCAMFNet(in_channels=1, enc_channels=256, enc_kernel_size=80, enc_stride=40,
                 chunk_size=500, hop_size=250, n_blocks=10, n_heads=8,
                 ffn_hidden=512, dw_kernel_size=31).to(device)

ckpt = torch.load(PROJECT_ROOT / 'experiments' / 'dcamf_net' / 'checkpoints' / 'best_SISNR.pth',
                  map_location=device, weights_only=False)
sd = {k: v for k, v in ckpt.items()
      if not k.endswith('.total_ops') and not k.endswith('.total_params')
      and k not in ('total_ops', 'total_params')}
model.load_state_dict(sd)
model.eval()

# Load a sample
cf = sorted((PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1' / 'clean').glob('*.wav'))[0]
clean, sr = sf.read(str(cf))
noisy, _ = sf.read(str(PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1' / 'noisy' / cf.name))
if clean.ndim > 1: clean = clean.mean(1)
if noisy.ndim > 1: noisy = noisy.mean(1)
L = 48000; clean, noisy = clean[:L], noisy[:L]
noisy_t = torch.from_numpy(noisy).float().unsqueeze(0).unsqueeze(0).to(device)

print(f'Loaded: {sum(p.numel() for p in model.parameters()):,} params')
print(f'Sample: {L/sr:.1f}s, {sr} Hz')


## 1. 输入长什么样？

In [ ]:
import IPython.display as ipd

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
t = np.arange(L) / sr

axes[0].plot(t*1000, noisy, '#E74C3C', lw=0.3)
axes[0].set_title('Noisy Waveform'); axes[0].set_xlabel('Time (ms)'); axes[0].set_xlim(0, 300)

f, p = welch(noisy, sr, nperseg=1024)
axes[1].semilogy(f, p, '#E74C3C', lw=0.8)
axes[1].set_title('PSD'); axes[1].set_xlabel('Hz'); axes[1].set_xlim(0, 4000)

f_s, t_s, Sxx = spectrogram(noisy, sr, nperseg=256, noverlap=200)
axes[2].pcolormesh(t_s, f_s, 10*np.log10(Sxx+1e-10), shading='auto', cmap='plasma')
axes[2].set_title('Spectrogram'); axes[2].set_ylim(0, 4000); axes[2].set_xlabel('Time (s)')
plt.tight_layout(); plt.show()

print(f'RMS amplitude: {np.sqrt(np.mean(noisy**2)):.4f}')
print('Listen:'); ipd.display(ipd.Audio(noisy, rate=sr))


## 2. 编码器把信号分解成了什么？

In [ ]:
# Register hooks for one forward pass
A = {}
def hk(name):
    def hook(module, input, output):
        if isinstance(output, torch.Tensor): A[name] = output.detach().cpu()
        elif isinstance(output, (list, tuple)): A[name] = [o.detach().cpu() if isinstance(o, torch.Tensor) else o for o in output]
    return hook

hooks = []
hooks.append(model.encoder.conv.register_forward_hook(hk('enc_conv')))
hooks.append(model.encoder.prelu.register_forward_hook(hk('enc_prelu')))
for b in range(10):
    blk = model.dcam_blocks[b]
    hooks.append(blk.register_forward_hook(hk(f'b{b}')))
    hooks.append(blk.global_cemhsa.pw_conv1.register_forward_hook(hk(f'g{b}')))
    hooks.append(blk.local_cemhsa.pw_conv1.register_forward_hook(hk(f'l{b}')))
hooks.append(model.decoder.register_forward_hook(hk('dec')))

with torch.no_grad():
    enhanced = model(noisy_t)
enhanced_np = enhanced.squeeze().cpu().numpy()

# Show 4 representative filters + their frequency responses
weights = model.encoder.conv.weight.detach().cpu().squeeze(1).numpy()  # (256, 80)
from numpy.fft import fft, fftfreq
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
freq = fftfreq(1024, 1/sr)
for i, fi in enumerate([10, 50, 100, 180]):
    axes[0,i].plot(np.arange(80)/sr*1000, weights[fi], '#1C7293', lw=0.7)
    axes[0,i].set_title(f'Filter #{fi}'); axes[0,i].set_xlabel('ms')
    resp = np.abs(fft(weights[fi], n=1024))
    axes[1,i].plot(freq[:512], resp[:512], '#E67E22', lw=0.8)
    axes[1,i].set_title(f'Freq Response #{fi}'); axes[1,i].set_xlabel('Hz')
    axes[1,i].set_xlim(0, 4000)
plt.tight_layout(); plt.show()

# Pre vs Post PReLU (activations)
pre = A['enc_conv'][0, :16].numpy()
post = A['enc_prelu'][0, :16].numpy()
t_enc = np.arange(pre.shape[1]) * 40 / sr * 1000
vmax = max(abs(pre).max(), abs(post).max())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(pre, aspect='auto', origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
               extent=[t_enc[0], t_enc[-1], 0, 16])
axes[0].set_title('Before PReLU'); axes[0].set_xlabel('Time (ms)'); axes[0].set_ylabel('Ch')
axes[1].imshow(post, aspect='auto', origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
               extent=[t_enc[0], t_enc[-1], 0, 16])
axes[1].set_title('After PReLU (negatives kept)'); axes[1].set_xlabel('Time (ms)'); axes[1].set_ylabel('Ch')
plt.tight_layout(); plt.show()
print(f'Encoded: {pre.shape[1]} frames (stride=40, from {L} samples)')


## 3. 局部 vs 全局注意力 — 各自捕获什么？

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for col, b in enumerate([0, 4, 9]):
    g = A.get(f'g{b}'); l = A.get(f'l{b}')
    if isinstance(g, torch.Tensor) and g.ndim >= 2:
        gf = g[0, :, :min(80, g.shape[-1])].numpy()
        im = axes[0,col].imshow(gf[:16], aspect='auto', cmap='RdBu_r')
        axes[0,col].set_title(f'Block {b}: Global (cross-chunk)'); axes[0,col].set_xlabel('Chunk idx')
        plt.colorbar(im, ax=axes[0,col])
    if isinstance(l, torch.Tensor) and l.ndim >= 2:
        lf = l[0, :, :min(80, l.shape[-1])].numpy()
        im = axes[1,col].imshow(lf[:16], aspect='auto', cmap='RdBu_r')
        axes[1,col].set_title(f'Block {b}: Local (within-chunk)'); axes[1,col].set_xlabel('Time in chunk')
        plt.colorbar(im, ax=axes[1,col])

plt.tight_layout(); plt.show()
print('Top row: Global — smooth, captures long-range trends.')
print('Bottom row: Local — detailed, preserves fine temporal structure.')
print('Shallow (Block 0): rich detail. Deep (Block 9): abstract patterns.')


## 4. GLU 门控 — 信号如何被调制？

In [ ]:
g0 = A.get('g0'); l0 = A.get('l0')
if g0 is not None and l0 is not None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 7))
    for col, (data, name) in enumerate([(g0, 'Global'), (l0, 'Local')]):
        ch = data.shape[1]; half = ch // 2
        sig = data[0, :half, :].numpy()
        gate = torch.sigmoid(data[0, half:, :]).numpy()

        im = axes[0,col].imshow(sig[:16, :min(80, sig.shape[-1])], aspect='auto', cmap='RdBu_r')
        axes[0,col].set_title(f'{name}: H1 (signal)'); axes[0,col].set_xlabel('Seq')
        plt.colorbar(im, ax=axes[0,col])

        im = axes[1,col].imshow(gate[:16, :min(80, gate.shape[-1])], aspect='auto',
                                 cmap='RdYlGn', vmin=0, vmax=1)
        axes[1,col].set_title(f'{name}: sigmoid(H2) (gate)'); axes[1,col].set_xlabel('Seq')
        plt.colorbar(im, ax=axes[1,col])
    plt.tight_layout(); plt.show()
    print('GLU: output = H1 * sigmoid(H2)')
    print('Green in bottom row = gate open (keep). Red/dark = gate closed (suppress).')


## 5. 10层掩码 — 每层贡献了什么？

In [ ]:
masks = {}
for b in range(10):
    out = A.get(f'b{b}')
    if out is not None and isinstance(out, (list, tuple)) and len(out) >= 2:
        m = out[1]
        if isinstance(m, torch.Tensor): masks[b] = m

if masks:
    # Show 4 representative masks: blocks 0, 3, 6, 9
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for i, b in enumerate([0, 3, 6, 9]):
        if b in masks:
            m = masks[b][0, :16, :min(50, masks[b].shape[-1])].numpy()
            im = axes[i].imshow(m, aspect='auto', cmap='RdBu_r')
            axes[i].set_title(f'Block {b} Mask'); axes[i].set_xlabel('T'); axes[i].set_ylabel('Ch')
            if i == 3: plt.colorbar(im, ax=axes[i])
    plt.tight_layout(); plt.show()

    # Energy trend across all 10
    fig, ax = plt.subplots(figsize=(10, 3.5))
    energy = [torch.norm(masks[b]).item() for b in sorted(masks.keys())]
    ax.bar(range(10), energy, color='#1C7293', edgecolor='black', lw=0.5)
    ax.set_xlabel('Block'); ax.set_ylabel('||Mask||_2')
    ax.set_title('Mask Energy per Block'); ax.grid(axis='y', alpha=0.3)
    ax.set_xticks(range(10))
    plt.tight_layout(); plt.show()

    fw = ckpt.get('mask_fusion_weights')
    if fw is not None:
        fw = fw.cpu().numpy() if isinstance(fw, torch.Tensor) else np.array(fw)
        fig, ax = plt.subplots(figsize=(10, 3.5))
        ax.bar(range(1, 11), fw, color='#E67E22', edgecolor='black', lw=0.5)
        for i, w in enumerate(fw):
            ax.text(i+1, w, f'{w:.2f}', ha='center', va='bottom', fontsize=7)
        ax.set_xlabel('Layer'); ax.set_ylabel('Fusion Weight')
        ax.set_title('Mask Fusion Weights (Softmax) — fixed after training')
        ax.set_xticks(range(1, 11)); ax.grid(axis='y', alpha=0.3)
        plt.tight_layout(); plt.show()
        print(f'Dominant layers: {np.argsort(fw)[-3:][::-1]+1}')


## 6. 模型估计的噪声是什么？

In [ ]:
dec = A.get('dec')
if isinstance(dec, torch.Tensor):
    n_est = dec.squeeze().numpy()[:L]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    t = np.arange(L) / sr * 1000

    axes[0].fill_between(t[:2000], 0, n_est[:2000], color='#E67E22', alpha=0.4)
    axes[0].plot(t[:2000], n_est[:2000], '#E67E22', lw=0.4)
    axes[0].set_title('Estimated Noise n(t)'); axes[0].set_xlabel('Time (ms)')

    f_n, p_n = welch(noisy, sr, nperseg=1024)
    _, p_ne = welch(n_est, sr, nperseg=1024)
    fm = (f_n >= 0) & (f_n <= 4000)
    axes[1].semilogy(f_n[fm], p_n[fm], 'gray', lw=0.5, alpha=0.6, label='Noisy input')
    axes[1].semilogy(f_n[fm], p_ne[fm], '#E67E22', lw=0.8, label='Estimated noise')
    axes[1].set_title('Noise PSD'); axes[1].legend(fontsize=7)
    axes[1].set_xlabel('Hz')

    axes[2].pie([np.mean(clean**2), np.mean(n_est**2), np.mean((noisy-clean-n_est)**2)],
                labels=['Signal', 'Noise removed', 'Residual'],
                colors=['#27AE60', '#E67E22', '#bbb'], autopct='%1.1f%%')
    axes[2].set_title('Energy Split')
    plt.tight_layout(); plt.show()

    print(f'Noise removed: {np.mean(n_est**2)/np.mean(noisy**2)*100:.1f}% of input energy')
    print('Estimated noise:'); ipd.display(ipd.Audio(n_est, rate=sr))


## 7. 降噪结果 — 跟干净信号差多少？

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
t = np.arange(L) / sr

# Waveform
for ax, sig, c, lbl in [(axes[0,0], noisy, '#E74C3C', 'Noisy'),
                          (axes[0,0], enhanced_np, '#1C7293', 'Enhanced'),
                          (axes[0,0], clean, 'k', 'Clean')]:
    ax.plot(t[:5000], sig[:5000], c, lw=0.3, label=lbl, alpha=0.7)
axes[0,0].set_title('Waveform (~300ms)'); axes[0,0].legend(fontsize=7)
axes[0,0].set_xlabel('Time (s)'); axes[0,0].set_xlim(0, 0.3)

# PSD
f_c, p_c = welch(clean, sr, nperseg=1024)
f_n, p_n = welch(noisy, sr, nperseg=1024)
f_e, p_e = welch(enhanced_np, sr, nperseg=1024)
fm = (f_c >= 0) & (f_c <= 4000)
axes[0,1].plot(f_c[fm]/1000, 10*np.log10(p_c[fm]), 'k', lw=1.2, label='Clean')
axes[0,1].plot(f_n[fm]/1000, 10*np.log10(p_n[fm]), '#E74C3C', lw=0.5, alpha=0.6, label='Noisy')
axes[0,1].plot(f_e[fm]/1000, 10*np.log10(p_e[fm]), '#1C7293', lw=1.0, label='Enhanced')
axes[0,1].set_title('PSD'); axes[0,1].set_xlabel('kHz'); axes[0,1].legend(fontsize=7)
axes[0,1].grid(alpha=0.3)

# Spectrograms
f_s, t_s, Sn = spectrogram(noisy, sr, nperseg=256, noverlap=200)
_, _, Se = spectrogram(enhanced_np, sr, nperseg=256, noverlap=200)
for ax, Sxx, title in [(axes[1,0], Sn, 'Noisy'), (axes[1,1], Se, 'Enhanced')]:
    ax.pcolormesh(t_s, f_s, 10*np.log10(Sxx+1e-10), shading='auto', cmap='plasma')
    ax.set_title(title); ax.set_ylim(0, 4000); ax.set_xlabel('Time (s)')
plt.tight_layout(); plt.show()

# Audio
print('Noisy:'); ipd.display(ipd.Audio(noisy, rate=sr))
print('Enhanced:'); ipd.display(ipd.Audio(enhanced_np, rate=sr))
print('Clean:'); ipd.display(ipd.Audio(clean, rate=sr))


## 8. 全流程 — 信号走了多远

In [ ]:
# Compute SI-SNRi
def sisnr(est, ref):
    e = est.reshape(-1); r = ref.reshape(-1)
    e = e - e.mean(); r = r - r.mean()
    s = np.dot(e, r) * r / (np.dot(r, r) + 1e-8)
    return 10*np.log10(np.sum(s**2)/(np.sum((e-s)**2)+1e-8)+1e-8)

sii = sisnr(enhanced_np, clean) - sisnr(noisy, clean)
print(f'SI-SNRi = {sii:.2f} dB')

# Energy flow through key stages
stages = ['Input', 'Encoded', 'Block 0', 'Block 4', 'Block 9', 'Noise est', 'Output']
vals = [
    np.sqrt(np.mean(noisy**2)),
    float(torch.norm(A['enc_prelu']).item() / 1000),
    float(torch.norm(A['b0'][0] if isinstance(A['b0'],(list,tuple)) else A['b0']).item()/1000),
    float(torch.norm(A['b4'][0] if isinstance(A['b4'],(list,tuple)) else A['b4']).item()/1000),
    float(torch.norm(A['b9'][0] if isinstance(A['b9'],(list,tuple)) else A['b9']).item()/1000),
    float(torch.norm(A['dec']).item()/1000) if isinstance(A['dec'], torch.Tensor) else 0,
    np.sqrt(np.mean(enhanced_np**2)),
]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(7), vals, color=['#E74C3C']+['#1C7293']*5+['#27AE60'], edgecolor='black', lw=0.5)
ax.set_xticks(range(7)); ax.set_xticklabels(stages, rotation=30)
ax.set_ylabel('RMS Energy (scaled)'); ax.set_title('Signal Energy Flow')
for i, v in enumerate(vals):
    ax.text(i, v, f'{v:.1f}', ha='center', va='bottom', fontsize=7)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

print('Pipeline: x -> Encoder -> 10 DCAM blocks -> Mask fusion -> Decoder -> n_hat')
print('Output: s_hat = x - n_hat')
